# TF-IDF Text Baseline (IEMOCAP)

Train a text-only baseline from IEMOCAP transcriptions using a session split
(Sessions 1-4 train, Session 5 test).


In [10]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Resolve dataset paths relative to repository root.
repo_root = Path.cwd().parents[1]
meta_csv = repo_root / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
iemocap_root = repo_root / "datasets" / "IEMOCAP"
meta_csv


WindowsPath('f:/Speech-Emotion-Recognition/datasets/IEMOCAP/iemocap_full_dataset.csv')

In [11]:
df = pd.read_csv(meta_csv)
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()
# Keep rows that are labeled (not xxx) and have at least some annotator agreement.
df = df[(df["emotion"] != "xxx") & (df["agreement"] > 0)].copy()
df.shape


(7532, 7)

In [12]:
# Parse transcript lines like:
# Ses01F_script02_1_F000 [015.1400-017.2100]: Fine.
# Groups:
# - utt: utterance ID token before the timestamp
# - text: spoken text after ":"
line_re = re.compile(r"^(?P<utt>\S+)\s+\[[^\]]+\]:\s*(?P<text>.*)$")

def build_transcript_index(iemocap_dir: Path) -> dict[str, str]:
    # Build utterance_id -> transcript text by scanning all transcription files.
    idx: dict[str, str] = {}
    for txt in iemocap_dir.glob("Session*/dialog/transcriptions/*.txt"):
        with txt.open("r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                m = line_re.match(line.strip())
                if not m:
                    continue
                utt = m.group("utt")
                text = m.group("text").strip()
                if not text:
                    continue
                # Some utterances may appear multiple times; concatenate to preserve content.
                if utt in idx:
                    idx[utt] = (idx[utt] + " " + text).strip()
                else:
                    idx[utt] = text
    return idx

transcripts = build_transcript_index(iemocap_root)
len(transcripts)


10084

In [13]:
# Derive utterance ID from wav path stem and join with transcript text.
df["utt_id"] = df["path"].apply(lambda p: Path(p).stem)
df["text"] = df["utt_id"].map(transcripts)

missing_text = df["text"].isna().sum()
print(f"Rows with missing transcript text: {missing_text}")

# Keep only rows with non-empty text for text-only modeling.
df_text = df[df["text"].notna() & (df["text"].str.len() > 0)].copy()
df_text.shape


Rows with missing transcript text: 0


(7532, 9)

In [14]:
# Session-based split to match existing audio experiments.
train_mask = df_text["session"].isin([1, 2, 3, 4])
test_mask = df_text["session"] == 5

X_train_text = df_text.loc[train_mask, "text"]
y_train = df_text.loc[train_mask, "emotion"]
X_test_text = df_text.loc[test_mask, "text"]
y_test = df_text.loc[test_mask, "emotion"]

X_train_text.shape, X_test_text.shape


((5882,), (1650,))

In [15]:
# TF-IDF baseline: unigrams+bigrams, drop very rare terms.
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
)

# Linear classifier baseline for sparse TF-IDF features.
clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="liblinear",
    random_state=42,
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1:  {f1:.4f}")
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 0.5067
Macro F1:  0.3455

Classification report:

              precision    recall  f1-score   support

         ang       0.51      0.67      0.58       170
         dis       0.00      0.00      0.00         0
         exc       0.57      0.41      0.48       299
         fea       0.15      0.30      0.20        10
         fru       0.57      0.59      0.58       381
         hap       0.41      0.31      0.35       143
         neu       0.53      0.47      0.49       384
         oth       0.00      0.00      0.00         0
         sad       0.57      0.56      0.56       245
         sur       0.13      0.72      0.22        18

    accuracy                           0.51      1650
   macro avg       0.34      0.40      0.35      1650
weighted avg       0.53      0.51      0.51      1650



f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [16]:
# Cross-validation on the train split only.
pipe = make_pipeline(
    TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_features=20000),
    LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear", random_state=42),
)

min_class_count = int(y_train.value_counts().min())
n_splits = max(2, min(5, min_class_count))
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print(f"Using StratifiedKFold with n_splits={n_splits}")
cv_acc = cross_val_score(pipe, X_train_text, y_train, cv=skf, scoring="accuracy", n_jobs=-1)
cv_f1 = cross_val_score(pipe, X_train_text, y_train, cv=skf, scoring="f1_macro", n_jobs=-1)

print(f"CV Accuracy (mean+-std): {np.mean(cv_acc):.4f} +- {np.std(cv_acc):.4f}")
print(f"CV Macro F1 (mean+-std):  {np.mean(cv_f1):.4f} +- {np.std(cv_f1):.4f}")


f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


CV Accuracy (mean+-std): 0.5099 +- 0.0093
CV Macro F1 (mean+-std):  0.3960 +- 0.0206


## Comparison: Baseline vs Stopwords + Stemming

This section compares the current TF-IDF baseline against a variant that removes English stopwords and applies Porter stemming.


In [21]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

try:
    from nltk.stem import PorterStemmer
except ImportError as e:
    raise ImportError("nltk is required for stemming. Install it with: pip install nltk") from e

stemmer = PorterStemmer()
token_re = re.compile(r"(?u)\b\w\w+\b")
stop_words = set(ENGLISH_STOP_WORDS)

def stemming_only_tokenizer(doc: str) -> list[str]:
    # Tokenize + stem while keeping stopwords.
    tokens = token_re.findall(doc.lower())
    return [stemmer.stem(t) for t in tokens]

def stemmed_stopword_tokenizer(doc: str) -> list[str]:
    # Tokenize, remove stopwords, then stem.
    tokens = token_re.findall(doc.lower())
    return [stemmer.stem(t) for t in tokens if t not in stop_words]

baseline_pipe = make_pipeline(
    TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_features=20000),
    LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear", random_state=42),
)

stopwords_only_pipe = make_pipeline(
    TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000,
    ),
    LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear", random_state=42),
)

stemming_only_pipe = make_pipeline(
    TfidfVectorizer(
        tokenizer=stemming_only_tokenizer,
        token_pattern=None,
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000,
    ),
    LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear", random_state=42),
)

stopwords_stemming_pipe = make_pipeline(
    TfidfVectorizer(
        tokenizer=stemmed_stopword_tokenizer,
        token_pattern=None,
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000,
    ),
    LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear", random_state=42),
)

results = []
for name, pipe in [
    ("baseline", baseline_pipe),
    ("stopwords_only", stopwords_only_pipe),
    ("stemming_only", stemming_only_pipe),
    ("stopwords+stemming", stopwords_stemming_pipe),
]:
    pipe.fit(X_train_text, y_train)
    y_pred = pipe.predict(X_test_text)
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred, average="macro")

    cv_acc = cross_val_score(pipe, X_train_text, y_train, cv=skf, scoring="accuracy", n_jobs=-1)
    cv_f1 = cross_val_score(pipe, X_train_text, y_train, cv=skf, scoring="f1_macro", n_jobs=-1)

    results.append(
        {
            "model": name,
            "test_acc": test_acc,
            "test_macro_f1": test_f1,
            "cv_acc_mean": float(np.mean(cv_acc)),
            "cv_acc_std": float(np.std(cv_acc)),
            "cv_macro_f1_mean": float(np.mean(cv_f1)),
            "cv_macro_f1_std": float(np.std(cv_f1)),
        }
    )

results_df = pd.DataFrame(results).sort_values("test_macro_f1", ascending=False).reset_index(drop=True)
display(results_df)

for _, row in results_df.iterrows():
    print(
        f"{row['model']}: CV Accuracy={row['cv_acc_mean']:.4f} +- {row['cv_acc_std']:.4f}, "
        f"CV Macro F1={row['cv_macro_f1_mean']:.4f} +- {row['cv_macro_f1_std']:.4f}"
    )


f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised

,model,test_acc,test_macro_f1,cv_acc_mean,cv_acc_std,cv_macro_f1_mean,cv_macro_f1_std
0,stemming_only,0.515758,0.349285,0.510368,0.007667,0.395052,0.023745
1,baseline,0.506667,0.345528,0.509858,0.009265,0.395983,0.020568
2,stopwords_only,0.459394,0.302468,0.461407,0.008667,0.329600,0.025375
3,stopwords+stemming,0.458788,0.302011,0.461236,0.005579,0.328777,0.020759


stemming_only: CV Accuracy=0.5104 +- 0.0077, CV Macro F1=0.3951 +- 0.0237
baseline: CV Accuracy=0.5099 +- 0.0093, CV Macro F1=0.3960 +- 0.0206
stopwords_only: CV Accuracy=0.4614 +- 0.0087, CV Macro F1=0.3296 +- 0.0254
stopwords+stemming: CV Accuracy=0.4612 +- 0.0056, CV Macro F1=0.3288 +- 0.0208


## Comparison on Selected Emotions (hap, exc, fru, neu, ang)

Repeat the same TF-IDF comparison using only the five target emotions.


In [22]:
selected_emotions = ["hap", "exc", "fru", "neu", "ang"]
df_sel = df_text[df_text["emotion"].isin(selected_emotions)].copy()

train_mask_sel = df_sel["session"].isin([1, 2, 3, 4])
test_mask_sel = df_sel["session"] == 5

X_train_text_sel = df_sel.loc[train_mask_sel, "text"]
y_train_sel = df_sel.loc[train_mask_sel, "emotion"]
X_test_text_sel = df_sel.loc[test_mask_sel, "text"]
y_test_sel = df_sel.loc[test_mask_sel, "emotion"]

print("Train class counts:")
print(y_train_sel.value_counts().sort_index())

min_class_count_sel = int(y_train_sel.value_counts().min())
n_splits_sel = max(2, min(5, min_class_count_sel))
skf_sel = StratifiedKFold(n_splits=n_splits_sel, shuffle=True, random_state=42)
print(f"Using StratifiedKFold with n_splits={n_splits_sel}")

baseline_pipe_sel = make_pipeline(
    TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_features=20000),
    LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear", random_state=42),
)

stopwords_only_pipe_sel = make_pipeline(
    TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=2, max_features=20000),
    LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear", random_state=42),
)

stemming_only_pipe_sel = make_pipeline(
    TfidfVectorizer(
        tokenizer=stemming_only_tokenizer,
        token_pattern=None,
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000,
    ),
    LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear", random_state=42),
)

stopwords_stemming_pipe_sel = make_pipeline(
    TfidfVectorizer(
        tokenizer=stemmed_stopword_tokenizer,
        token_pattern=None,
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000,
    ),
    LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear", random_state=42),
)

results_sel = []
for name, pipe in [
    ("baseline", baseline_pipe_sel),
    ("stopwords_only", stopwords_only_pipe_sel),
    ("stemming_only", stemming_only_pipe_sel),
    ("stopwords+stemming", stopwords_stemming_pipe_sel),
]:
    pipe.fit(X_train_text_sel, y_train_sel)
    y_pred_sel = pipe.predict(X_test_text_sel)
    test_acc_sel = accuracy_score(y_test_sel, y_pred_sel)
    test_f1_sel = f1_score(y_test_sel, y_pred_sel, average="macro")

    cv_acc_sel = cross_val_score(pipe, X_train_text_sel, y_train_sel, cv=skf_sel, scoring="accuracy", n_jobs=-1)
    cv_f1_sel = cross_val_score(pipe, X_train_text_sel, y_train_sel, cv=skf_sel, scoring="f1_macro", n_jobs=-1)

    results_sel.append(
        {
            "model": name,
            "test_acc": test_acc_sel,
            "test_macro_f1": test_f1_sel,
            "cv_acc_mean": float(np.mean(cv_acc_sel)),
            "cv_acc_std": float(np.std(cv_acc_sel)),
            "cv_macro_f1_mean": float(np.mean(cv_f1_sel)),
            "cv_macro_f1_std": float(np.std(cv_f1_sel)),
        }
    )

results_sel_df = pd.DataFrame(results_sel).sort_values("test_macro_f1", ascending=False).reset_index(drop=True)
display(results_sel_df)

for _, row in results_sel_df.iterrows():
    print(
        f"{row['model']}: CV Accuracy={row['cv_acc_mean']:.4f} +- {row['cv_acc_std']:.4f}, "
        f"CV Macro F1={row['cv_macro_f1_mean']:.4f} +- {row['cv_macro_f1_std']:.4f}"
    )


Train class counts:
emotion
ang     933
exc     742
fru    1468
hap     452
neu    1324
Name: count, dtype: int64
Using StratifiedKFold with n_splits=5


f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use anothe

,model,test_acc,test_macro_f1,cv_acc_mean,cv_acc_std,cv_macro_f1_mean,cv_macro_f1_std
0,stemming_only,0.559187,0.538391,0.567189,0.010095,0.548293,0.012544
1,baseline,0.555556,0.535409,0.567798,0.010340,0.548564,0.010738
2,stopwords+stemming,0.519245,0.503352,0.517583,0.011602,0.506915,0.009202
3,stopwords_only,0.515614,0.499886,0.516162,0.009514,0.504291,0.007837


stemming_only: CV Accuracy=0.5672 +- 0.0101, CV Macro F1=0.5483 +- 0.0125
baseline: CV Accuracy=0.5678 +- 0.0103, CV Macro F1=0.5486 +- 0.0107
stopwords+stemming: CV Accuracy=0.5176 +- 0.0116, CV Macro F1=0.5069 +- 0.0092
stopwords_only: CV Accuracy=0.5162 +- 0.0095, CV Macro F1=0.5043 +- 0.0078
